<a href="https://colab.research.google.com/github/yuzonfire907/data-science-2026/blob/main/Pertemuan%2010_Yustinus%20Budi%20Kristiawan_240401010299.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritma Klasifikasi (Bagian 2)
- **Nama Lengkap**: Yustinus Budi Kristiawan
- **NIM**: 240401010299
- **Kelas**: IF401

**1. Import Dataset**

---



In [ ]:
import urllib.request
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
urllib.request.urlretrieve(url, "telco_churn.csv")
print("Dataset berhasil diunduh.")

Dataset berhasil diunduh.


**2. Eksplorasi Data**

In [ ]:
import pandas as pd

df = pd.read_csv("telco_churn.csv")
print("Shape:", df.shape)
print("\nProporsi kelas Churn:")
print(df["Churn"].value_counts(normalize=True).round(3))
df.head()

Shape: (7043, 21)

Proporsi kelas Churn:
Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


**3. Processing Data**

In [ ]:
if "customerID" in df.columns:
    df = df.drop(columns=["customerID"])

if "TotalCharges" in df.columns and df["TotalCharges"].dtype == "object":
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

if df["Churn"].dtype == "object":
    df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

X = df.drop(columns=["Churn"])
y = df["Churn"]

X = pd.get_dummies(X, drop_first=True)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)  # [cite: 101, 202]

print("=== Proses Preprocessing Selesai ===")
print(f"Dimensi X_train: {X_tr.shape}, Dimensi X_test: {X_te.shape}")

=== Proses Preprocessing Selesai ===
Dimensi X_train: (5634, 30), Dimensi X_test: (1409, 30)


**4. Latih Model**

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300, class_weight="balanced", random_state=42
)  # [cite: 208]

rf.fit(X_tr, y_tr)  # [cite: 209]

print("Model Random Forest berhasil dilatih dengan class_weight='balanced'.")

Model Random Forest berhasil dilatih dengan class_weight='balanced'.


**5. Evaluasi**

In [ ]:
y_pred = rf.predict(X_te)

y_proba = rf.predict_proba(X_te)[:, 1]  # [cite: 109]

print("=== Classification Report ===")
print(classification_report(y_te, y_pred))  # [cite: 214]

print("=== ROC-AUC Score ===")
print(f"ROC-AUC: {roc_auc_score(y_te, y_proba):.4f}")  # [cite: 214]

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

=== ROC-AUC Score ===
ROC-AUC: 0.8246


**6.Prediksi Probabilitas & Kesimpulan**

In [ ]:
df_hasil = pd.DataFrame(
    {"Aktual": y_te, "Prediksi": y_pred, "Probabilitas_Churn": y_proba}
)

print("=== Sampel Hasil Prediksi Probabilitas ===")
print(df_hasil.head())

print("\n=== Kesimpulan Analisis ===")
kesimpulan = (
    "1. Mengingat distribusi data telco churn ini bersifat tidak seimbang (imbalanced), penerapan parameter "
    "class_weight='balanced' pada Random Forest terbukti sangat krusial. Pendekatan ini berhasil meningkatkan "
    "sensitivitas model dalam mendeteksi pelanggan yang berpotensi berhenti berlangganan, yang ditunjukkan oleh nilai Recall yang optimal.\n"
    "2. Dalam analisis ini, metrik akurasi sengaja tidak dijadikan acuan utama demi menghindari jebakan 'accuracy paradox'. "
    "Fokus evaluasi dialihkan pada kombinasi nilai Recall dan F1-Score untuk memastikan kelas minoritas (churn) tetap terprediksi dengan akurat.\n"
    "3. Kinerja pemeringkatan model juga menunjukkan hasil yang memuaskan berdasarkan skor ROC-AUC yang diperoleh. Hal ini "
    "menandakan bahwa algoritma yang dibangun memiliki kapabilitas yang baik dalam memisahkan karakteristik antara pelanggan yang loyal dan yang berisiko.\n"
    "4. Melalui pemanfaatan fungsi 'predict_proba', perusahaan tidak hanya mendapatkan label prediksi mutlak, melainkan juga "
    "nilai probabilitas spesifik dari setiap pelanggan. Output berupa persentase risiko ini sangat berguna bagi tim retensi untuk mengambil tindakan preventif yang tepat sasaran."
)
print(kesimpulan)

=== Sampel Hasil Prediksi Probabilitas ===
      Aktual  Prediksi  Probabilitas_Churn
437        0         0            0.000000
2280       0         1            0.786667
2235       0         0            0.090000
4460       0         0            0.280000
3761       0         0            0.000000

=== Kesimpulan Analisis ===
1. Mengingat distribusi data telco churn ini bersifat tidak seimbang (imbalanced), penerapan parameter class_weight='balanced' pada Random Forest terbukti sangat krusial. Pendekatan ini berhasil meningkatkan sensitivitas model dalam mendeteksi pelanggan yang berpotensi berhenti berlangganan, yang ditunjukkan oleh nilai Recall yang optimal.
2. Dalam analisis ini, metrik akurasi sengaja tidak dijadikan acuan utama demi menghindari jebakan 'accuracy paradox'. Fokus evaluasi dialihkan pada kombinasi nilai Recall dan F1-Score untuk memastikan kelas minoritas (churn) tetap terprediksi dengan akurat.
3. Kinerja pemeringkatan model juga menunjukkan hasil yang memuaskan 